# 04. 함수 심화

기본값 인자의 함정, `*args`/`**kwargs`, 키워드 전용 인자, 람다, 클로저까지.

## 기본값 인자의 함정: mutable default

**가장 유명한 파이썬 버그 패턴.** 기본값은 함수 정의 시점에 '한 번만' 평가되어 재사용됩니다. 리스트/딕셔너리를 기본값으로 쓰면 호출 간에 상태가 공유됩니다.

In [1]:
def add_item_buggy(item, bucket=[]):  # 절대 이렇게 쓰면 안 됨
    bucket.append(item)
    return bucket

print(add_item_buggy("a"))
print(add_item_buggy("b"))  # "a"가 남아있음! 기대와 다름

def add_item_safe(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket

print(add_item_safe("a"))
print(add_item_safe("b"))  # 매번 새 리스트


['a']
['a', 'b']
['a']
['b']


## *args, **kwargs

`*args`는 위치 인자를 튜플로, `**kwargs`는 키워드 인자를 딕셔너리로 모읍니다.

In [2]:
def log_call(func_name, *args, **kwargs):
    print(f"{func_name} 호출: args={args}, kwargs={kwargs}")

log_call("predict", 1, 2, model="resnet", threshold=0.5)

def merge_configs(**overrides):
    base = {"lr": 0.001, "batch_size": 32}
    base.update(overrides)
    return base

print(merge_configs(lr=0.01, epochs=5))


predict 호출: args=(1, 2), kwargs={'model': 'resnet', 'threshold': 0.5}
{'lr': 0.01, 'batch_size': 32, 'epochs': 5}


## 키워드 전용 인자 (`*` 뒤는 반드시 키워드로)

인자가 많아지면 호출부에서 순서를 헷갈리기 쉬운데, `*`를 넣으면 뒤의 인자는 반드시 이름을 명시해야 해서 실수를 방지합니다.

In [3]:
def train(model, *, lr=0.001, epochs=10):
    print(f"model={model}, lr={lr}, epochs={epochs}")

train("resnet", lr=0.01, epochs=20)  # OK
try:
    train("resnet", 0.01, 20)  # 에러: 위치 인자로 못 넘김
except TypeError as e:
    print("에러:", e)


model=resnet, lr=0.01, epochs=20
에러: train() takes 1 positional argument but 3 were given


## lambda: 익명 함수

한 줄짜리 간단한 함수에만 쓰는 게 좋습니다. `sorted`의 `key`처럼 짧은 콜백이 필요할 때 유용.

In [4]:
people = [("alice", 30), ("bob", 25), ("carol", 35)]
print(sorted(people, key=lambda p: p[1]))  # 나이순 정렬

square = lambda x: x ** 2
print(square(5))


[('bob', 25), ('alice', 30), ('carol', 35)]
25


## 클로저 (closure)

함수가 자신을 둘러싼 스코프의 변수를 '기억'하는 것. `nonlocal`로 바깥 스코프 변수를 수정할 수 있습니다.

In [5]:
def make_counter():
    count = 0
    def increment():
        nonlocal count
        count += 1
        return count
    return increment

counter_a = make_counter()
counter_b = make_counter()
print(counter_a(), counter_a(), counter_a())  # 1 2 3
print(counter_b())  # 1 — 서로 독립적인 상태를 가짐


1 2 3
1


## 일급 함수 (first-class function)

함수를 변수에 담거나 다른 함수의 인자로 넘길 수 있습니다. 데코레이터(05번)의 기반이 되는 개념.

In [6]:
def apply_twice(func, value):
    return func(func(value))

print(apply_twice(lambda x: x * 2, 3))  # 3 -> 6 -> 12


12


## 헷갈렸던 점 / 더 찾아볼 것

- mutable default 함정은 코드 리뷰에서 자주 지적되는 포인트라 꼭 기억
- 클로저는 05번 데코레이터, 11번 비동기 콜백에서 다시 등장함